# NYC Yellow Taxi - data exploration

View and profile the raw TLC files (`data/input/`) and the cleaned pipeline output (`data/output/`).
Use the project `.venv` as the kernel (`ipykernel` is in `requirements/dev.txt`). Change `MONTH` to look at another month.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)  # show every column
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = ROOT / "data" / "input"
OUTPUT_DIR = ROOT / "data" / "output"
MONTH = "2023-02"

## Available files

## Raw trips: view the data

In [3]:
raw = pd.read_parquet(INPUT_DIR / f"yellow_tripdata_{MONTH}.parquet")
print(f"{len(raw):,} rows x {raw.shape[1]} columns")
raw.head(10)

2,913,955 rows x 19 columns


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2023-02-01 00:32:53,2023-02-01 00:34:34,2.0,0.30,1.0,N,142,163,2,4.4,3.50,0.5,0.00,0.0,1.0,9.40,2.5,0.00
1,2,2023-02-01 00:35:16,2023-02-01 00:35:30,1.0,0.00,1.0,N,71,71,4,-3.0,-1.00,-0.5,0.00,0.0,-1.0,-5.50,0.0,0.00
2,2,2023-02-01 00:35:16,2023-02-01 00:35:30,1.0,0.00,1.0,N,71,71,4,3.0,1.00,0.5,0.00,0.0,1.0,5.50,0.0,0.00
3,1,2023-02-01 00:29:33,2023-02-01 01:01:38,0.0,18.80,1.0,N,132,26,1,70.9,2.25,0.5,0.00,0.0,1.0,74.65,0.0,1.25
4,2,2023-02-01 00:12:28,2023-02-01 00:25:46,1.0,3.22,1.0,N,161,145,1,17.0,1.00,0.5,3.30,0.0,1.0,25.30,2.5,0.00
5,1,2023-02-01 00:52:40,2023-02-01 01:07:18,1.0,5.10,1.0,N,148,236,1,21.9,3.50,0.5,5.35,0.0,1.0,32.25,2.5,0.00
6,1,2023-02-01 00:12:39,2023-02-01 00:40:36,1.0,8.90,1.0,N,137,244,1,41.5,3.50,0.5,3.50,0.0,1.0,50.00,2.5,0.00
7,1,2023-02-01 00:56:53,2023-02-01 01:00:37,1.0,1.20,1.0,N,263,141,1,7.2,3.50,0.5,2.44,0.0,1.0,14.64,2.5,0.00
8,2,2023-02-01 00:20:40,2023-02-01 00:33:56,1.0,7.49,1.0,N,48,243,1,30.3,1.00,0.5,8.82,0.0,1.0,44.12,2.5,0.00
9,2,2023-02-01 00:33:51,2023-02-01 00:37:34,1.0,0.51,1.0,N,114,211,1,5.8,1.00,0.5,1.62,0.0,1.0,12.42,2.5,0.00


In [4]:
raw.tail(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2913950,2,2023-02-28 23:46:00,2023-03-01 00:05:00,NaN,4.65,NaN,NaN,249,140,0,20.22,0.0,0.5,4.84,0.0,1.0,29.06,NaN,NaN
2913951,2,2023-02-28 23:26:02,2023-02-28 23:37:10,NaN,2.47,NaN,NaN,186,79,0,13.66,0.0,0.5,2.65,0.0,1.0,20.31,NaN,NaN
2913952,2,2023-02-28 23:24:00,2023-02-28 23:38:00,NaN,3.49,NaN,NaN,158,143,0,17.64,0.0,0.5,0.00,0.0,1.0,21.64,NaN,NaN
2913953,2,2023-02-28 23:03:00,2023-02-28 23:10:00,NaN,2.13,NaN,NaN,79,162,0,13.56,0.0,0.5,2.63,0.0,1.0,20.19,NaN,NaN
2913954,2,2023-02-28 23:03:03,2023-02-28 23:12:51,NaN,2.28,NaN,NaN,161,140,0,14.89,0.0,0.5,3.78,0.0,1.0,22.67,NaN,NaN


In [ ]:
raw.sample(10, random_state=42)  # random rows from anywhere in the file

In [ ]:
raw.dtypes

In [ ]:
raw.isna().sum()

In [ ]:
raw.describe().T

In [ ]:
raw.payment_type.value_counts()

### Rows the cleaning rules would reject

In [ ]:
suspect = raw[
    (raw.trip_distance <= 0)
    | (raw.fare_amount <= 0)
    | (raw.tpep_dropoff_datetime <= raw.tpep_pickup_datetime)
]
print(f"{len(suspect):,} suspect rows")
suspect.head(10)

## Taxi zone lookup

In [ ]:
zones = pd.read_csv(INPUT_DIR / "taxi_zone_lookup.csv")
print(f"{len(zones)} zones")
zones.head(10)

In [ ]:
zones.Borough.value_counts()

## Cleaned output (after the `clean` task)

In [ ]:
clean_path = OUTPUT_DIR / f"yellow_tripdata_clean_{MONTH}.parquet"
if clean_path.exists():
    clean = pd.read_parquet(clean_path)
    print(f"{len(clean):,} rows kept, {len(raw) - len(clean):,} rejected ({1 - len(clean) / len(raw):.2%})")
    display(clean.head(10))
else:
    print(f"{clean_path.relative_to(ROOT)} not found: run the pipeline for {MONTH} first")

### Cleaned trips with zone names

In [ ]:
if clean_path.exists():
    named = clean.merge(
        zones.add_prefix("pickup_"), left_on="pickup_location_id", right_on="pickup_LocationID", how="left"
    )
    display(named[["pickup_ts", "dropoff_ts", "pickup_Borough", "pickup_Zone", "trip_distance", "fare_amount", "total_amount"]].head(10))